In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Load Dataset
df = pd.read_csv('ted_talks.csv')


In [ ]:
shortlisted_cols = [
    'socialTitle', 'description', 'duration', 'publishedAt', 
    'videoContext', 'viewedCount', 'relatedVideos', 'topics', 
    'type', 'speakers'
]
df_clean = df[[col for col in shortlisted_cols if col in df.columns]].drop_duplicates(subset=['socialTitle', 'description']).copy()

def parse_topics(val):
    try:
        data = ast.literal_eval(val) if isinstance(val, str) else val
        if isinstance(data, list):
            return [item['name'] for item in data if isinstance(item, dict) and 'name' in item]
    except Exception:
        pass
    return []

def parse_speaker(val):
    try:
        data = ast.literal_eval(val) if isinstance(val, str) else val
        if isinstance(data, list) and len(data) > 0:
            sp = data[0]
            name = f"{sp.get('firstName', '')} {sp.get('lastName', '')}".strip()
            return name if name else 'Unknown Speaker'
    except Exception:
        pass
    return 'Unknown Speaker'

def parse_type(val):
    try:
        data = ast.literal_eval(val) if isinstance(val, str) else val
        if isinstance(data, dict):
            return data.get('name', 'TED Stage Talk')
    except Exception:
        pass
    return 'TED Stage Talk'

df_clean['topic_list'] = df_clean['topics'].apply(parse_topics)
df_clean['topic_tags'] = df_clean['topic_list'].apply(lambda x: ', '.join(x))
df_clean['topic_count'] = df_clean['topic_list'].apply(len)
df_clean['speaker_name'] = df_clean['speakers'].apply(parse_speaker)
df_clean['talk_type'] = df_clean['type'].apply(parse_type)

df_clean['publishedAt_dt'] = pd.to_datetime(df_clean['publishedAt'], errors='coerce')
ref_date = df_clean['publishedAt_dt'].max()
df_clean['published_year'] = df_clean['publishedAt_dt'].dt.year.fillna(2015).astype(int)
df_clean['published_month'] = df_clean['publishedAt_dt'].dt.month.fillna(1).astype(int)
df_clean['published_dayofweek'] = df_clean['publishedAt_dt'].dt.dayofweek.fillna(0).astype(int)
df_clean['talk_age_days'] = (ref_date - df_clean['publishedAt_dt']).dt.days.fillna(0).astype(int)

df_clean['duration_mins'] = (df_clean['duration'] / 60.0).round(2)
df_clean['title_word_count'] = df_clean['socialTitle'].fillna('').apply(lambda x: len(str(x).split()))
df_clean['desc_word_count'] = df_clean['description'].fillna('').apply(lambda x: len(str(x).split()))

top_contexts = df_clean['videoContext'].value_counts().nlargest(8).index.tolist()
df_clean['event_category'] = df_clean['videoContext'].apply(lambda x: x if x in top_contexts else 'Other')

# Popularity Target
median_views = df_clean['viewedCount'].median()
df_clean['is_popular'] = (df_clean['viewedCount'] >= median_views).astype(int)

num_features = ['duration_mins', 'talk_age_days', 'published_year', 'published_month', 'published_dayofweek', 'topic_count', 'title_word_count', 'desc_word_count']
cat_features = ['event_category', 'talk_type']

X = df_clean[num_features + cat_features]
y = df_clean['is_popular']


In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features), 
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

# Fit tuned models directly
lr_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(C=10.0, max_iter=200, solver='liblinear', random_state=42))])
lr_pipe.fit(X, y)

dt_pipe = Pipeline([('prep', preprocessor), ('clf', DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=50, random_state=42))])
dt_pipe.fit(X, y)

rf_pipe = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(max_depth=12, max_features='sqrt', min_samples_leaf=2, n_estimators=150, random_state=42))])
rf_pipe.fit(X, y)

# TF-IDF
df_clean['combined_content'] = (
    df_clean['socialTitle'].fillna('') + ' ' + 
    df_clean['description'].fillna('') + ' ' + 
    df_clean['topic_tags'].fillna('') + ' ' + 
    df_clean['speaker_name'].fillna('')
)

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df_clean['combined_content'])


In [ ]:
class MultiModelTEDRecommender:
    def __init__(self, data_df, tfidf_model, tfidf_mat, lr_model, dt_model, rf_model, feature_cols):
        self.df = data_df.reset_index(drop=True).copy()
        self.tfidf = tfidf_model
        self.tfidf_matrix = tfidf_mat
        self.lr_model = lr_model
        self.dt_model = dt_model
        self.rf_model = rf_model
        self.feature_cols = feature_cols
        
        X_all = self.df[self.feature_cols]
        
        self.df['lr_prob'] = self.lr_model.predict_proba(X_all)[:, 1]
        self.df['lr_pred'] = self.lr_model.predict(X_all)
        
        self.df['dt_prob'] = self.dt_model.predict_proba(X_all)[:, 1]
        self.df['dt_pred'] = self.dt_model.predict(X_all)
        
        self.df['rf_prob'] = self.rf_model.predict_proba(X_all)[:, 1]
        self.df['rf_pred'] = self.rf_model.predict(X_all)
        
        self.watched_talk_ids = set()
        self.dynamic_view_counter = {i: 0 for i in range(len(self.df))}
        
    def record_interaction(self, talk_id):
        """Logs user interaction, increments view count, and adds to watch history."""
        talk_id = int(talk_id)
        if talk_id in self.dynamic_view_counter:
            self.watched_talk_ids.add(talk_id)
            self.dynamic_view_counter[talk_id] += 1

    def get_candidate_talks(self, query, candidate_pool_size=30, filter_viewed=True):
        """Stage 1: Retrieve top semantically relevant talks using TF-IDF & Cosine Similarity."""
        query_vec = self.tfidf.transform([query])
        sim_scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        
        sorted_indices = sim_scores.argsort()[::-1]
        candidates = []
        for idx in sorted_indices:
            if sim_scores[idx] <= 0:
                break
            if filter_viewed and idx in self.watched_talk_ids:
                continue
            candidates.append(idx)
            if len(candidates) >= candidate_pool_size:
                break
        return candidates, sim_scores

    def recommend_by_model(self, query, model_choice='rf', top_n=6, filter_viewed=True):
        """Stage 2: Rank retrieved candidates by the chosen model's popularity probability."""
        candidate_ids, sim_scores = self.get_candidate_talks(query, filter_viewed=filter_viewed)
        if not candidate_ids:
            return pd.DataFrame()
        cand_df = self.df.iloc[candidate_ids].copy()
        cand_df['similarity'] = [sim_scores[i] for i in candidate_ids]
        prob_col = f'{model_choice}_prob'
        pred_col = f'{model_choice}_pred'
        ranked_df = cand_df.sort_values(by=[prob_col, 'similarity'], ascending=[False, False]).head(top_n)
        results = []
        for rank, (_, row) in enumerate(ranked_df.iterrows(), 1):
            idx = int(row.name)
            self.dynamic_view_counter[idx] += 1
            results.append({
                'Rank': rank,
                'ID': idx,
                'Title': row['socialTitle'],
                'Speaker': row['speaker_name'],
                'Event': row['videoContext'],
                'Duration': f"{row['duration_mins']}m",
                'Topic Relevance (Sim)': round(float(row['similarity']), 3),
                'Popularity Prob': f"{round(float(row[prob_col]) * 100, 1)}%",
                'Predicted Class': 'High' if row[pred_col] == 1 else 'Low',
                'Session Views': self.dynamic_view_counter[idx]
            })
        return pd.DataFrame(results)

    def compare_models_on_candidates(self, query, top_n=6, ranking_model='rf', filter_viewed=True):
        """Compares how all 3 models evaluate the recommended talks side-by-side."""
        candidate_ids, sim_scores = self.get_candidate_talks(query, filter_viewed=filter_viewed)
        if not candidate_ids:
            return pd.DataFrame()
        cand_df = self.df.iloc[candidate_ids].copy()
        cand_df['similarity'] = [sim_scores[i] for i in candidate_ids]
        if ranking_model in ['rf', 'lr', 'dt']:
            ranked_df = cand_df.sort_values(by=[f'{ranking_model}_prob', 'similarity'], ascending=[False, False]).head(top_n)
        else:
            ranked_df = cand_df.sort_values(by='similarity', ascending=False).head(top_n)
        comp_rows = []
        for rank, (_, row) in enumerate(ranked_df.iterrows(), 1):
            idx = int(row.name)
            preds = [row['lr_pred'], row['dt_pred'], row['rf_pred']]
            consensus = 'Full Agreement' if (preds[0] == preds[1] == preds[2]) else 'Majority'
            comp_rows.append({
                'Rank': rank,
                'ID': idx,
                'Title': row['socialTitle'][:40] + '...' if len(row['socialTitle']) > 40 else row['socialTitle'],
                'Speaker': row['speaker_name'],
                'Cosine Relevance': round(float(row['similarity']), 3),
                'Logistic Reg': f"{'High' if row['lr_pred']==1 else 'Low'} ({round(row['lr_prob']*100, 1)}%)",
                'Decision Tree': f"{'High' if row['dt_pred']==1 else 'Low'} ({round(row['dt_prob']*100, 1)}%)",
                'Random Forest': f"{'High' if row['rf_pred']==1 else 'Low'} ({round(row['rf_prob']*100, 1)}%)",
                'Consensus': consensus
            })
        return pd.DataFrame(comp_rows)

    def display_html_dashboard(self, query, top_n=3, ranking_model='rf', filter_viewed=True):
        """Renders an elegant, self-contained HTML visual dashboard across all 3 models + comparison table."""
        from IPython.display import display, HTML
        
        df_lr = self.recommend_by_model(query, model_choice='lr', top_n=top_n, filter_viewed=filter_viewed)
        df_dt = self.recommend_by_model(query, model_choice='dt', top_n=top_n, filter_viewed=filter_viewed)
        df_rf = self.recommend_by_model(query, model_choice='rf', top_n=top_n, filter_viewed=filter_viewed)
        comp_df = self.compare_models_on_candidates(query, top_n=top_n+3, ranking_model=ranking_model, filter_viewed=filter_viewed)
        
        def build_cards(df, border_color):
            if df.empty:
                return "<p style='color:#6c757d; font-style:italic; padding:10px;'>No unwatched talks found.</p>"
            html = ""
            for _, r in df.iterrows():
                pop_color = "#198754" if "High" in str(r['Predicted Class']) else "#fd7e14"
                html += f"""
                <div style='background:#ffffff; border:1px solid #dee2e6; border-left:4px solid {border_color}; border-radius:6px; padding:10px 12px; margin-bottom:8px; box-shadow:0 1px 3px rgba(0,0,0,0.05);'>
                    <div style='display:flex; justify-content:space-between; font-size:11px; margin-bottom:4px;'>
                        <span style='background:#f1f3f5; color:#495057; padding:1px 5px; border-radius:3px;'>#{r['Rank']} | {r['Event']} ({r['Duration']})</span>
                        <span style='color:{pop_color}; font-weight:bold;'>{r['Predicted Class']} ({r['Popularity Prob']})</span>
                    </div>
                    <div style='font-size:13px; font-weight:bold; color:#212529; line-height:1.2; margin-bottom:4px;'>{r['Title']}</div>
                    <div style='color:#eb0028; font-size:11px; font-weight:600;'>Speaker: {r['Speaker']}</div>
                    <div style='font-size:11px; color:#6c757d; margin-top:4px;'>Cosine Relevance: <b>{r['Topic Relevance (Sim)']}</b> | Views: <b>{r['Session Views']}</b></div>
                </div>
                """
            return html

        lr_cards = build_cards(df_lr, "#0d6efd")
        dt_cards = build_cards(df_dt, "#198754")
        rf_cards = build_cards(df_rf, "#6f42c1")
        
        comp_table_html = comp_df.to_html(classes='table table-striped table-hover', index=False, border=0)
        
        total_views = sum(self.dynamic_view_counter.values())
        watched_count = len(self.watched_talk_ids)
        
        dashboard_html = f"""
        <div style='font-family:-apple-system,BlinkMacSystemFont,\"Segoe UI\",Roboto,sans-serif; background:#f8f9fa; border:1px solid #ced4da; border-radius:10px; padding:16px; margin:12px 0;'>
            <div style='background:#212529; color:#ffffff; padding:12px 16px; border-radius:8px; margin-bottom:14px; display:flex; justify-content:space-between; align-items:center;'>
                <div>
                    <span style='font-size:16px; font-weight:bold;'>Multi-Model TED Recommendation Showcase</span>
                    <div style='font-size:12px; color:#adb5bd;'>Topic Query: <span style='color:#ffd166; font-weight:bold;'>\"{query}\"</span></div>
                </div>
                <div style='font-size:11px; text-align:right;'>
                    <div>Watched Filter: <b>{'Active' if filter_viewed else 'Off'}</b> ({watched_count} marked)</div>
                    <div>Active Session Views: <b>{total_views}</b></div>
                </div>
            </div>
            
            <div style='display:grid; grid-template-columns:1fr 1fr 1fr; gap:12px; margin-bottom:16px;'>
                <div style='background:#ffffff; border:1px solid #dee2e6; border-radius:8px; padding:12px;'>
                    <div style='font-weight:bold; color:#0d6efd; font-size:13px; border-bottom:2px solid #0d6efd; padding-bottom:4px; margin-bottom:10px;'>
                        1. Logistic Regression
                        <div style='font-size:10px; color:#6c757d; font-weight:normal;'>Linear Probability Baseline (AUC: 0.60)</div>
                    </div>
                    {lr_cards}
                </div>
                <div style='background:#ffffff; border:1px solid #dee2e6; border-radius:8px; padding:12px;'>
                    <div style='font-weight:bold; color:#198754; font-size:13px; border-bottom:2px solid #198754; padding-bottom:4px; margin-bottom:10px;'>
                        2. Decision Tree
                        <div style='font-size:10px; color:#6c757d; font-weight:normal;'>Tuned Pruned Decision Tree (AUC: 0.72)</div>
                    </div>
                    {dt_cards}
                </div>
                <div style='background:#ffffff; border:1px solid #dee2e6; border-radius:8px; padding:12px;'>
                    <div style='font-weight:bold; color:#6f42c1; font-size:13px; border-bottom:2px solid #6f42c1; padding-bottom:4px; margin-bottom:10px;'>
                        3. Random Forest
                        <div style='font-size:10px; color:#6c757d; font-weight:normal;'>Tuned Bagging Ensemble (AUC: 0.74)</div>
                    </div>
                    {rf_cards}
                </div>
            </div>
            
            <div style='background:#ffffff; border:1px solid #dee2e6; border-radius:8px; padding:12px;'>
                <div style='font-weight:bold; color:#212529; font-size:13px; margin-bottom:8px;'>
                    Side-by-Side Consensus & Comparison Across Recommended Talks:
                </div>
                <div style='overflow-x:auto;'>
                    {comp_table_html}
                </div>
            </div>
        </div>
        """
        display(HTML(dashboard_html))

recommender = MultiModelTEDRecommender(
    data_df=df_clean,
    tfidf_model=tfidf,
    tfidf_mat=tfidf_matrix,
    lr_model=lr_pipe,
    dt_model=dt_pipe,
    rf_model=rf_pipe,
    feature_cols=num_features + cat_features
)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
class MultiModelWidgetApp:
    def __init__(self, recommender_system):
        self.rec = recommender_system
        self._build_interface()
        
    def _build_interface(self):
        self.query_input = widgets.Text(
            value='artificial intelligence',
            placeholder='Type any topic: e.g. climate change, psychology, robotics...',
            description='Topic:',
            continuous_update=False,
            layout=widgets.Layout(width='65%')
        )
        self.search_btn = widgets.Button(
            description='Find Talks',
            button_style='danger',
            layout=widgets.Layout(width='120px')
        )
        self.top_n_slider = widgets.IntSlider(
            value=4, min=2, max=10, step=1,
            description='Top Talks:',
            layout=widgets.Layout(width='280px')
        )
        self.filter_checkbox = widgets.Checkbox(
            value=True,
            description='Filter Watched Talks (Interaction-Aware)',
            layout=widgets.Layout(width='320px')
        )
        self.reset_btn = widgets.Button(
            description='Reset Session',
            button_style='warning',
            layout=widgets.Layout(width='140px')
        )
        self.preset_labels = ['AI & Robotics', 'Climate & Earth', 'Psychology & Mind', 'Future Tech', 'Leadership']
        self.preset_queries = [
            'artificial intelligence robotics',
            'climate change environment nature',
            'psychology human mind happiness',
            'future technology space internet',
            'leadership business success innovation'
        ]
        self.preset_buttons = []
        for lbl, qry in zip(self.preset_labels, self.preset_queries):
            btn = widgets.Button(description=lbl, button_style='info', layout=widgets.Layout(height='28px', margin='2px 4px'))
            btn.on_click(lambda b, q=qry: self._on_preset_click(q))
            self.preset_buttons.append(btn)
        
        self.status_box = widgets.HTML(value=self._status_html())
        
        self.out_rf = widgets.Output()
        self.out_lr = widgets.Output()
        self.out_dt = widgets.Output()
        self.out_comp = widgets.Output()
        
        self.tabs = widgets.Tab(children=[self.out_rf, self.out_lr, self.out_dt, self.out_comp])
        self.tabs.set_title(0, 'Random Forest')
        self.tabs.set_title(1, 'Logistic Regression')
        self.tabs.set_title(2, 'Decision Tree')
        self.tabs.set_title(3, 'Multi-Model Comparison')
        
        self.search_btn.on_click(lambda b: self.run_search())
        self.query_input.observe(lambda change: self.run_search(), names='value')
        self.reset_btn.on_click(lambda b: self.reset_session())
        self.top_n_slider.observe(lambda change: self.run_search(), names='value')
        self.filter_checkbox.observe(lambda change: self.run_search(), names='value')
        
        header_banner = widgets.HTML("""
        <div style='background:linear-gradient(135deg, #1f2937, #111827); border-radius:10px; padding:14px 18px; color:white; margin-bottom:8px;'>
            <div style='font-size:18px; font-weight:bold; color:#ffffff;'>
                <span style='color:#eb0028;'>TED</span> Interactive Multi-Model Recommender Prototype
            </div>
            <div style='font-size:12px; color:#9ca3af; margin-top:3px;'>
                Two-Stage Architecture: TF-IDF Cosine Retrieval + Supervised Popularity Re-Ranking (Logistic Regression, Decision Tree, Random Forest)
            </div>
        </div>
        """)
        
        input_row = widgets.HBox([self.query_input, self.search_btn, self.reset_btn], layout=widgets.Layout(align_items='center', margin='4px 0'))
        presets_row = widgets.HBox([widgets.Label('Quick Topics:')] + self.preset_buttons, layout=widgets.Layout(align_items='center', margin='2px 0 6px 0'))
        controls_row = widgets.HBox([self.top_n_slider, self.filter_checkbox], layout=widgets.Layout(align_items='center', margin='4px 0'))
        
        self.gui = widgets.VBox([
            header_banner,
            self.status_box,
            input_row,
            presets_row,
            controls_row,
            self.tabs
        ], layout=widgets.Layout(padding='10px', border='1px solid #dee2e6', border_radius='10px', background_color='#f8f9fa'))
        
    def _on_preset_click(self, query):
        self.query_input.value = query
        self.run_search()

    def _status_html(self):
        total_views = sum(self.rec.dynamic_view_counter.values())
        watched_count = len(self.rec.watched_talk_ids)
        return f"""
            <div style='display:flex; gap:16px; font-size:12px; background:#e9ecef; padding:6px 12px; border-radius:6px; margin-bottom:8px;'>
                <span><b>Live Session Views:</b> <span style='color:#0d6efd; font-weight:bold;'>{total_views}</span></span>
                <span><b>Watched History:</b> <span style='color:#198754; font-weight:bold;'>{watched_count} talks</span></span>
                <span><b>Popularity Benchmark:</b> Median Views Threshold (1,362,484)</span>
            </div>
        """

    def mark_and_refresh(self, talk_id):
        self.rec.record_interaction(talk_id)
        self.status_box.value = self._status_html()
        self.run_search(status_msg=f"Talk #{talk_id} marked as watched. Filtered across all models.")

    def reset_session(self):
        self.rec.watched_talk_ids.clear()
        self.rec.dynamic_view_counter = {i: 0 for i in range(len(self.rec.df))}
        self.status_box.value = self._status_html()
        self.run_search(status_msg="Session state reset. Watched history cleared.")

    def render_model_tab(self, out_widget, model_code, model_name, query, top_n, filter_w, status_msg):
        df_res = self.rec.recommend_by_model(query, model_choice=model_code, top_n=top_n, filter_viewed=filter_w)
        with out_widget:
            clear_output(wait=True)
            if status_msg:
                display(HTML(f"<div style='background:#d1e7dd; color:#0f5132; padding:8px 12px; border-radius:6px; margin-bottom:10px; font-size:12px;'><b>Update:</b> {status_msg}</div>"))
            
            if df_res.empty:
                display(HTML("<div style='padding:20px; color:#6c757d; text-align:center;'>No unwatched talks found for this query.</div>"))
                return
                
            for _, r in df_res.iterrows():
                tid = int(r['ID'])
                pop_color = "#198754" if "High" in str(r['Predicted Class']) else "#fd7e14"
                
                card_html = f"""
                <div style='background:#ffffff; border:1px solid #e9ecef; border-left:4px solid {pop_color}; border-radius:8px; padding:12px 16px; margin-bottom:10px; box-shadow:0 2px 5px rgba(0,0,0,0.05);'>
                    <div style='display:flex; justify-content:space-between; margin-bottom:4px;'>
                        <span style='font-size:11px; background:#f1f3f5; color:#495057; padding:2px 6px; border-radius:4px;'>{r['Event']} | Duration: {r['Duration']}</span>
                        <span style='font-size:12px; font-weight:bold; color:{pop_color};'>{r['Predicted Class']} Popularity ({r['Popularity Prob']})</span>
                    </div>
                    <div style='font-size:15px; font-weight:bold; color:#212529;'>{r['Title']}</div>
                    <div style='color:#eb0028; font-size:13px; font-weight:600; margin-bottom:6px;'>Speaker: {r['Speaker']}</div>
                    <div style='font-size:12px; color:#6c757d;'>Topic Cosine Relevance: <b>{r['Topic Relevance (Sim)']}</b> | Session Views: <b>{r['Session Views']}</b></div>
                </div>
                """
                
                btn = widgets.Button(
                    description=f"Mark Talk #{tid} as Watched",
                    button_style='success',
                    layout=widgets.Layout(width='220px', height='28px', margin='-4px 0 12px 0')
                )
                btn.on_click(lambda b, t_id=tid: self.mark_and_refresh(t_id))
                
                display(HTML(card_html))
                display(btn)

    def render_comparison_tab(self, query, top_n, filter_w):
        comp_df = self.rec.compare_models_on_candidates(query, top_n=top_n, ranking_model='rf', filter_viewed=filter_w)
        with self.out_comp:
            clear_output(wait=True)
            if comp_df.empty:
                display(HTML("<div style='padding:20px; color:#6c757d; text-align:center;'>No candidates available.</div>"))
                return
            display(HTML("""
                <div style='margin-bottom:10px; font-size:13px; color:#495057;'>
                    <b>Side-by-Side Model Evaluation on Recommended Talks:</b>
                    Comparing how Logistic Regression, Decision Tree, and Random Forest evaluate candidate talks.
                </div>
            """))
            display(comp_df)

    def run_search(self, status_msg=None):
        q = self.query_input.value.strip()
        top_n = self.top_n_slider.value
        filter_w = self.filter_checkbox.value
        
        self.render_model_tab(self.out_rf, 'rf', 'Random Forest', q, top_n, filter_w, status_msg)
        self.render_model_tab(self.out_lr, 'lr', 'Logistic Regression', q, top_n, filter_w, status_msg)
        self.render_model_tab(self.out_dt, 'dt', 'Decision Tree', q, top_n, filter_w, status_msg)
        self.render_comparison_tab(q, top_n, filter_w)

    def display(self):
        display(self.gui)
        self.run_search()

try:
    app = MultiModelWidgetApp(recommender)
    app.display()
except Exception as e:
    print(f"Interactive widget notice: {e}")
    print("Rendering native visual presentation dashboard directly:")
    recommender.display_html_dashboard('artificial intelligence', top_n=4, ranking_model='rf')

def demo_recommend(topic="artificial intelligence", top_n=3, filter_viewed=True):
    """Call anytime during viva to see recommendations on any topic: demo_recommend('robotics')"""
    recommender.display_html_dashboard(topic, top_n=top_n, ranking_model='rf', filter_viewed=filter_viewed)

def mark_watched(talk_id):
    """Call to mark any talk as watched: mark_watched(123)"""
    recommender.record_interaction(talk_id)
    print(f"Talk #{talk_id} marked as watched. Total watched talks: {len(recommender.watched_talk_ids)}")
